In [ ]:
!pip install -q -U "trl==0.12.2" "transformers==4.46.3" "peft==0.13.2" "accelerate==1.0.1" bitsandbytes datasets huggingface_hub



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 100.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is in

In [ ]:
from google.colab import files
uploaded = files.upload()   # select day30_domain_corpus.jsonl here


Saving day30_domain_corpus.jsonl to day30_domain_corpus.jsonl


In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token)


Enter your Hugging Face token: ··········


In [ ]:
import torch, json
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig


## **Use Qwen Base Model**

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-1.5B"
DATA_PATH    = "/content/day30_domain_corpus.jsonl"
OUTPUT_DIR   = "/content/qwen2.5-1.5b-base-ai-safety-lora"
HUB_REPO     = "nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora"
MAX_SEQ_LEN  = 256


In [ ]:
import os

# Confirm exactly which file we're about to load, and its real, current size
print(f"DATA_PATH points to: {DATA_PATH}")
print(f"File exists: {os.path.exists(DATA_PATH)}")
print(f"File size: {os.path.getsize(DATA_PATH)} bytes")
print(f"Last modified: {os.path.getmtime(DATA_PATH)}")

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    line_count = sum(1 for _ in f)
print(f"Lines in source file: {line_count}")

EXPECTED_TOTAL = 700
if line_count != EXPECTED_TOTAL:
    raise ValueError(
        f"Expected {EXPECTED_TOTAL} lines in {DATA_PATH}, but found {line_count}. "
        f"Check /content/ for duplicate uploads like 'day30_domain_corpus (1).jsonl' "
        f"and make sure DATA_PATH points to the correct one."
    )

# Force a completely fresh load, ignoring any cached dataset from a prior run
dataset = load_dataset("json", data_files=DATA_PATH, split="train", download_mode="force_redownload")
dataset = dataset.train_test_split(test_size=100, seed=42)
train_ds, eval_ds = dataset["train"], dataset["test"]

print(f"Total examples loaded: {len(train_ds) + len(eval_ds)}")
print(f"Train examples: {len(train_ds)}")
print(f"Eval examples: {len(eval_ds)}")
print(train_ds[0])

DATA_PATH points to: /content/day30_domain_corpus.jsonl
File exists: True
File size: 490387 bytes
Last modified: 1789462700.4222481
Lines in source file: 700


Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]

Total examples loaded: 700
Train examples: 600
Eval examples: 100
{'text': 'Treating algorithms as trade secrets protects companies, such as search engines, where a transparent algorithm might reveal tactics to manipulate search rankings. This makes it difficult for researchers to conduct interviews or analysis to discover how algorithms function. Critics suggest that such secrecy can also obscure possible unethical methods used in producing or processing algorithmic output. Other critics, such as lawyer and activist Katarzyna Szymielewicz, have suggested that the lack of transparency is often disguised as a result of algorithmic complexity, shielding companies from disclosing or investigating its own algorithmic processes.'}


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # match compute_dtype to avoid the
    bnb_4bit_use_double_quant=True,          # dtype-mismatch errors you hit on Day 29
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    max_steps=75,                 # <-- num_train_epochs=3 ki jagah
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,              # <-- 10 se 5
    eval_strategy="steps",
    eval_steps=5,                 # <-- 25 se 5
    save_strategy="steps",        # <-- "epoch" se "steps" (kyunki max_steps use kar rahe, epoch-based save match nahi karega)
    save_steps=25,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

print(f"Raw train examples going into SFTTrainer: {len(train_ds)}")
print(f"Raw eval examples going into SFTTrainer: {len(eval_ds)}")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

# After packing, trainer.train_dataset holds the PACKED chunks (not raw examples) -
# this count will be smaller than 600 because short passages get packed together
# into fewer, denser 256-token blocks. This is expected, not data loss.
print(f"Packed train chunks (after packing=True): {len(trainer.train_dataset)}")
print(f"Packed eval chunks (after packing=True): {len(trainer.eval_dataset)}")

trainer.train()

Raw train examples going into SFTTrainer: 600
Raw eval examples going into SFTTrainer: 100


max_steps is given, it will override any value given in num_train_epochs
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:428: UserWarning: You passed `packing=True` to the SFTTrainer/SFTConfig, and you are training your model with `max_steps` strategy. The dataset will be iterated until the `max_steps` are reached.
  warnings.warn(


Packed train chunks (after packing=True): 315
Packed eval chunks (after packing=True): 55


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
5,3.016200,2.899199
10,2.927000,2.835753
15,2.900100,2.806577
20,2.984400,2.784907
25,2.718500,2.775321
30,2.721700,2.771990
35,2.628400,2.767578
40,2.839100,2.763727
45,2.530200,2.765567
50,2.521800,2.775892


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=75, training_loss=2.694476140340169, metrics={'train_runtime': 1865.2172, 'train_samples_per_second': 0.643, 'train_steps_per_second': 0.04, 'total_flos': 2443130933870592.0, 'train_loss': 2.694476140340169, 'epoch': 3.7974683544303796})

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print(f"Adapter pushed to https://huggingface.co/{HUB_REPO}")


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpo1nrpt38/tokenizer.json:   0%|          | 27.6kB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Adapter pushed to https://huggingface.co/nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B"
ADAPTER_REPO = "nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)

print("Loading fine-tuned model (base + LoRA adapter)...")
ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_REPO)

Loading base model...
Loading fine-tuned model (base + LoRA adapter)...


adapter_config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
domain_prompts = [
    "Deepfake technology has recently",
    "The biggest risk of generative AI misuse is",
    "Governments are responding to synthetic media by",
    "Content authenticity standards such as C2PA are designed to",
    "Researchers studying AI alignment have found that",
]

print("=" * 70)
print("STEP 3: DOMAIN-RELEVANT PREFIXES - Base vs Fine-tuned")
print("=" * 70)
for p in domain_prompts:
    print(f"\n--- Prompt: \"{p}\" ---")
    print("[BASE]      ", generate(base_model, p))
    print("[FINE-TUNED]", generate(ft_model, p))



STEP 3: DOMAIN-RELEVANT PREFIXES - Base vs Fine-tuned

--- Prompt: "Deepfake technology has recently" ---
[BASE]       Deepfake technology has recently become a popular topic in media and tech circles. The term “deepfakes” is derived from the words deep learning, which refers to AI algorithms that mimic natural human language patterns.
In this article we will explore what DeepFakes are, how they work, and why you should be wary of
[FINE-TUNED] Deepfake technology has recently emerged as a significant challenge for media and communication research. This paper presents the first systematic overview of existing work in this emerging area, by systematically reviewing 60 papers published between 2018–2024. The authors also identify gaps that need to be addressed by future research efforts

--- Prompt: "The biggest risk of generative AI misuse is" ---
[BASE]       The biggest risk of generative AI misuse is that we don't have a good understanding of its potential consequences. The current st

In [ ]:
ood_prompts = [
    "The capital of France is",
    "To solve a quadratic equation, you first need to",
    "Photosynthesis is the process by which plants",
]

print("\n" + "=" * 70)
print("STEP 4: OUT-OF-DOMAIN PROMPTS - Catastrophic Forgetting Check")
print("=" * 70)
for p in ood_prompts:
    print(f"\n--- Prompt: \"{p}\" ---")
    print("[BASE]      ", generate(base_model, p))
    print("[FINE-TUNED]", generate(ft_model, p))


STEP 4: OUT-OF-DOMAIN PROMPTS - Catastrophic Forgetting Check

--- Prompt: "The capital of France is" ---
[BASE]       The capital of France is Paris, and the country's largest city is ____.
A. London
B. Paris
C. Berlin
D. Rome
Answer:
Paris

When using a pressure cooker for cooking, which of the following statements about its advantages is incorrect?
A. The food inside will be cooked more quickly
[FINE-TUNED] The capital of France is Paris. The country's largest cities are Marseille, Lyon, Lille and Toulouse. The most populous city in metropolitan France is Marseilles; the most densely populated urban area (AUP) is Lyon - Saint-Étienne – Dijon with 1 million inhabitants out of a

--- Prompt: "To solve a quadratic equation, you first need to" ---
[BASE]       To solve a quadratic equation, you first need to identify the coefficients. The general form of a second degree polynomial is \\(ax^2 + bx + c = 0\\), where \\(a,b,c\\) are constants and \\(x\\) represents an unknown variable.\nT

# **Use Qwen Instruct Model**

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_PATH    = "/content/day30_domain_corpus.jsonl"   # matches the uploaded file from Cell 2
OUTPUT_DIR   = "/content/qwen2.5-1.5b-instruct-ai-safety-lora"
HUB_REPO     = "nooruiit-864/qwen2.5-1.5b-instruct-ai-safety-domain-lora"
MAX_SEQ_LEN  = 256   # short on purpose: our passages are ~29 words avg -> keeps VRAM/time low
# core structural difference between instruction tuning and domain adaptation:
# no prompt/response split here, just raw continuation text -> full-sequence loss.

# ------------------------------------------------------------
# CELL 5: Load dataset (with verification)
# ------------------------------------------------------------
import os

print(f"DATA_PATH points to: {DATA_PATH}")
print(f"File exists: {os.path.exists(DATA_PATH)}")
print(f"File size: {os.path.getsize(DATA_PATH)} bytes")
print(f"Last modified: {os.path.getmtime(DATA_PATH)}")

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    line_count = sum(1 for _ in f)
print(f"Lines in source file: {line_count}")

EXPECTED_TOTAL = 700
if line_count != EXPECTED_TOTAL:
    raise ValueError(
        f"Expected {EXPECTED_TOTAL} lines in {DATA_PATH}, but found {line_count}. "
        f"Check /content/ for duplicate uploads like 'day30_domain_corpus (1).jsonl' "
        f"and make sure DATA_PATH points to the correct one."
    )

dataset = load_dataset("json", data_files=DATA_PATH, split="train", download_mode="force_redownload")
dataset = dataset.train_test_split(test_size=100, seed=42)
train_ds, eval_ds = dataset["train"], dataset["test"]

print(f"Total examples loaded: {len(train_ds) + len(eval_ds)}")
print(f"Train examples: {len(train_ds)}")
print(f"Eval examples: {len(eval_ds)}")
print(train_ds[0])

# ------------------------------------------------------------
# CELL 6: Load base model in 4-bit + tokenizer
# ------------------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # match compute_dtype to avoid the
    bnb_4bit_use_double_quant=True,          # dtype-mismatch errors you hit on Day 29
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

# ------------------------------------------------------------
# CELL 7: Attach LoRA adapter
# ------------------------------------------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ------------------------------------------------------------
# CELL 8: Training config + train (with packed-chunk verification)
# ------------------------------------------------------------
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    max_steps=75,                 # <-- num_train_epochs=3 ki jagah
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,              # <-- 10 se 5
    eval_strategy="steps",
    eval_steps=5,                 # <-- 25 se 5
    save_strategy="steps",        # <-- "epoch" se "steps" (kyunki max_steps use kar rahe, epoch-based save match nahi karega)
    save_steps=25,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

print(f"Raw train examples going into SFTTrainer: {len(train_ds)}")
print(f"Raw eval examples going into SFTTrainer: {len(eval_ds)}")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print(f"Packed train chunks (after packing=True): {len(trainer.train_dataset)}")
print(f"Packed eval chunks (after packing=True): {len(trainer.eval_dataset)}")

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print(f"Adapter pushed to https://huggingface.co/{HUB_REPO}")


# ============================================================
# CELL 9: EVALUATION — Base vs Fine-tuned (Instruct variant)
# ============================================================
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_REPO = "nooruiit-864/qwen2.5-1.5b-instruct-ai-safety-domain-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)

print("Loading fine-tuned model (base + LoRA adapter)...")
ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_REPO)

def generate(model, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# ---- Domain-relevant prefixes (AI Safety / Deepfake / Misinformation) ----
domain_prompts = [
    "Deepfake technology has recently",
    "The biggest risk of generative AI misuse is",
    "Governments are responding to synthetic media by",
    "Content authenticity standards such as C2PA are designed to",
    "Researchers studying AI alignment have found that",
]

print("=" * 70)
print("STEP 3: DOMAIN-RELEVANT PREFIXES - Base vs Fine-tuned")
print("=" * 70)
for p in domain_prompts:
    print(f"\n--- Prompt: \"{p}\" ---")
    print("[BASE]      ", generate(base_model, p))
    print("[FINE-TUNED]", generate(ft_model, p))

# ---- Out-of-domain prompts (catastrophic forgetting check) ----
ood_prompts = [
    "The capital of France is",
    "To solve a quadratic equation, you first need to",
    "Photosynthesis is the process by which plants",
]

print("\n" + "=" * 70)
print("STEP 4: OUT-OF-DOMAIN PROMPTS - Catastrophic Forgetting Check")
print("=" * 70)
for p in ood_prompts:
    print(f"\n--- Prompt: \"{p}\" ---")
    print("[BASE]      ", generate(base_model, p))
    print("[FINE-TUNED]", generate(ft_model, p))

DATA_PATH points to: /content/day30_domain_corpus.jsonl
File exists: True
File size: 490387 bytes
Last modified: 1789489056.300557
Lines in source file: 700


Generating train split: 0 examples [00:00, ? examples/s]

Total examples loaded: 700
Train examples: 600
Eval examples: 100
{'text': 'Treating algorithms as trade secrets protects companies, such as search engines, where a transparent algorithm might reveal tactics to manipulate search rankings. This makes it difficult for researchers to conduct interviews or analysis to discover how algorithms function. Critics suggest that such secrecy can also obscure possible unethical methods used in producing or processing algorithmic output. Other critics, such as lawyer and activist Katarzyna Szymielewicz, have suggested that the lack of transparency is often disguised as a result of algorithmic complexity, shielding companies from disclosing or investigating its own algorithmic processes.'}


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

No files have been modified since last commit. Skipping to prevent empty commit.


Adapter pushed to https://huggingface.co/nooruiit-864/qwen2.5-1.5b-instruct-ai-safety-domain-lora
Loading base model...
Loading fine-tuned model (base + LoRA adapter)...


adapter_config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

STEP 3: DOMAIN-RELEVANT PREFIXES - Base vs Fine-tuned

--- Prompt: "Deepfake technology has recently" ---
[BASE]       Deepfake technology has recently become a popular topic in the news, with many high-profile celebrities and politicians using it to create fake videos. One of the most common ways that this is done involves taking one or more photos from an individual’s face (the "face" being their eyes, nose, mouth, etc.) and then
[FINE-TUNED] Deepfake technology has recently emerged as a significant challenge for media and communication research. It is expected that it will continue to evolve in the coming years, making traditional forms of content production increasingly difficult. To address this issue, researchers have begun exploring new methods of content creation using generative AI systems.

--- Prompt: "The biggest risk of generative AI misuse is" ---
[BASE]       The biggest risk of generative AI misuse is that it can be used to spread misinformation, hate speech and other h

In [ ]:
print(model.config._name_or_path if hasattr(model, 'config') else "check manually")

Qwen/Qwen2.5-1.5B-Instruct


# **Day 32 LLM Fine-Tuning Mini-Project & Delivery-- 20 prompt evaluation**

In [ ]:
!pip install -q -U bitsandbytes
!pip install -q -U torchao


import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 31.3 MB/s eta 0:00:00


In [1]:
EVAL_PROMPTS = [
    # ===== IN-DOMAIN (12) — AI safety / deepfake / misinformation =====
    {"id": 1, "category": "in-domain", "prompt": "Deepfake detection tools typically work by"},
    {"id": 2, "category": "in-domain", "prompt": "Synthetic media has changed journalism because"},
    {"id": 3, "category": "in-domain", "prompt": "One major challenge in fighting online misinformation is"},
    {"id": 4, "category": "in-domain", "prompt": "The EU AI Act aims to"},
    {"id": 5, "category": "in-domain", "prompt": "Voice cloning technology raises concerns about"},
    {"id": 6, "category": "in-domain", "prompt": "During elections, disinformation campaigns often"},
    {"id": 7, "category": "in-domain", "prompt": "AI alignment researchers study"},
    {"id": 8, "category": "in-domain", "prompt": "Content provenance and watermarking help by"},
    {"id": 9, "category": "in-domain", "prompt": "Social media platforms respond to deepfakes by"},
    {"id": 10, "category": "in-domain", "prompt": "A key limitation of current deepfake detectors is"},
    {"id": 11, "category": "in-domain", "prompt": "Generative adversarial networks are used to create deepfakes because"},
    {"id": 12, "category": "in-domain", "prompt": "Public trust in media has been affected by AI because"},

    # ===== OUT-OF-DOMAIN (8) — general knowledge / math / biology / coding =====
    {"id": 13, "category": "out-of-domain", "prompt": "The capital of Japan is"},
    {"id": 14, "category": "out-of-domain", "prompt": "To solve a quadratic equation, you first"},
    {"id": 15, "category": "out-of-domain", "prompt": "Photosynthesis is the process by which"},
    {"id": 16, "category": "out-of-domain", "prompt": "A simple Python function to reverse a string looks like"},
    {"id": 17, "category": "out-of-domain", "prompt": "The water cycle consists of the following stages:"},
    {"id": 18, "category": "out-of-domain", "prompt": "World War II ended in the year"},
    {"id": 19, "category": "out-of-domain", "prompt": "The human heart has four chambers, which are"},
    {"id": 20, "category": "out-of-domain", "prompt": "In economics, supply and demand determine"},
]

print(f"Total prompts: {len(EVAL_PROMPTS)} ({sum(1 for p in EVAL_PROMPTS if p['category']=='in-domain')} in-domain, {sum(1 for p in EVAL_PROMPTS if p['category']=='out-of-domain')} out-of-domain)")

Total prompts: 20 (12 in-domain, 8 out-of-domain)


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
base_model.generation_config.pad_token_id = tokenizer.eos_token_id

def generate(model, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return full_text[len(prompt):].strip()

results = []
for item in EVAL_PROMPTS:
    base_response = generate(base_model, item["prompt"])
    results.append({
        "id": item["id"], "category": item["category"], "prompt": item["prompt"],
        "base_response": base_response,
    })
    print(f"[{item['id']}] Base done.")

del base_model
import gc; gc.collect(); torch.cuda.empty_cache()
print("Base model generation complete.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[1] Base done.
[2] Base done.
[3] Base done.
[4] Base done.
[5] Base done.
[6] Base done.
[7] Base done.
[8] Base done.
[9] Base done.
[10] Base done.
[11] Base done.
[12] Base done.
[13] Base done.
[14] Base done.
[15] Base done.
[16] Base done.
[17] Base done.
[18] Base done.
[19] Base done.
[20] Base done.
Base model generation complete.


In [3]:
from huggingface_hub import snapshot_download

MERGED_MODEL_DIR = snapshot_download(
    repo_id="nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora",
    allow_patterns=["*.json", "*.safetensors", "*.txt", "merges.txt", "vocab.json"],
)
print("Downloaded to:", MERGED_MODEL_DIR)

# FIX: FP16 load karo, koi quantization nahi — base model ki precision se match karne ke liye
ft_model = AutoModelForCausalLM.from_pretrained(MERGED_MODEL_DIR, torch_dtype=torch.float16, device_map="auto")
ft_model.generation_config.pad_token_id = tokenizer.eos_token_id

for i, item in enumerate(EVAL_PROMPTS):
    ft_response = generate(ft_model, item["prompt"])
    results[i]["finetuned_response"] = ft_response
    print(f"[{item['id']}] Fine-tuned (FP16) done.")

del ft_model
gc.collect(); torch.cuda.empty_cache()
print("Fine-tuned model generation complete.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Downloaded to: /root/.cache/huggingface/hub/models--nooruiit-864--qwen2.5-1.5b-base-ai-safety-domain-lora/snapshots/d21bada208f97543fe8b1675302372c1e8605281


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[1] Fine-tuned (FP16) done.
[2] Fine-tuned (FP16) done.
[3] Fine-tuned (FP16) done.
[4] Fine-tuned (FP16) done.
[5] Fine-tuned (FP16) done.
[6] Fine-tuned (FP16) done.
[7] Fine-tuned (FP16) done.
[8] Fine-tuned (FP16) done.
[9] Fine-tuned (FP16) done.
[10] Fine-tuned (FP16) done.
[11] Fine-tuned (FP16) done.
[12] Fine-tuned (FP16) done.
[13] Fine-tuned (FP16) done.
[14] Fine-tuned (FP16) done.
[15] Fine-tuned (FP16) done.
[16] Fine-tuned (FP16) done.
[17] Fine-tuned (FP16) done.
[18] Fine-tuned (FP16) done.
[19] Fine-tuned (FP16) done.
[20] Fine-tuned (FP16) done.
Fine-tuned model generation complete.


In [4]:
df = pd.DataFrame(results)
# Scoring columns add karo — manually fill karogi 1-5 scale pe
for col in ["base_relevance", "base_coherence", "base_factual", "base_safety",
            "ft_relevance", "ft_coherence", "ft_factual", "ft_safety"]:
    df[col] = ""

df.to_csv("/content/day32_eval_scoring.csv", index=False)
df.head(20)


,id,category,prompt,base_response,finetuned_response,base_relevance,base_coherence,base_factual,base_safety,ft_relevance,ft_coherence,ft_factual,ft_safety
0,1,in-domain,Deepfake detection tools typically work by,analyzing the content of the video and compari...,identifying patterns in audio generated by dee...,,,,,,,,
1,2,in-domain,Synthetic media has changed journalism because,it is a new way of producing and consuming new...,it is easier to create and share than traditio...,,,,,,,,
2,3,in-domain,One major challenge in fighting online misinfo...,that it is difficult to distinguish between fa...,the difficulty in identifying the source of th...,,,,,,,,
3,4,in-domain,The EU AI Act aims to,protect citizens and consumers from the risks ...,establish a high level of protection for human...,,,,,,,,
4,5,in-domain,Voice cloning technology raises concerns about,privacy and security. Which of the following s...,the potential for deepfake pornography and the...,,,,,,,,
5,6,in-domain,"During elections, disinformation campaigns often","target specific groups, such as women, minorit...","target specific demographics, such as age, gen...",,,,,,,,
6,7,in-domain,AI alignment researchers study,"the ethical implications of AI systems, and th...",the design and deployment of AI systems to ens...,,,,,,,,
7,8,in-domain,Content provenance and watermarking help by,design\n\nThe need for provenance and watermar...,identifying the source of misinformation.,,,,,,,,
8,9,in-domain,Social media platforms respond to deepfakes by,implementing various measures to combat the sp...,"removing them from their platforms, removing t...",,,,,,,,
9,10,in-domain,A key limitation of current deepfake detectors is,that they are trained on a small number of lab...,that they are trained on a single speaker's vo...,,,,,,,,
